In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os, time
 
spark = (SparkSession.builder
    .appName("Preprocessing")
    .master("local[8]")
    .config("spark.driver.memory",           "12g")
    .config("spark.executor.memory",         "16g")
    .config("spark.sql.shuffle.partitions",  "16")
    .config("spark.shuffle.consolidateFiles","true")
    .config("spark.local.dir",              "/var/tmp/spark-s20426")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
 
PARQUET_DIR = "data_parquet"
CLEAN_DIR   = "data_clean"
REPORT_DIR  = "preprocessing_report"
os.makedirs(CLEAN_DIR,  exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
 

 


26/05/05 09:55:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 09:55:42 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


In [2]:
res  = spark.read.parquet(f"{PARQUET_DIR}/resource")
rtq  = spark.read.parquet(f"{PARQUET_DIR}/rtqps")
cg   = spark.read.parquet(f"{PARQUET_DIR}/callgraph")
 
TRAFFIC_COLS = [c for c in [
    "providerRPC_MCR","providerRPC_RT",
    "consumerRPC_MCR","consumerRPC_RT",
    "HTTP_MCR","HTTP_RT",
    "consumerMQ_MCR","consumerMQ_RT",
] if c in rtq.columns]
 
tables = {
    "MSResource":  (res,  ["cpu_utilization","memory_utilization",
                            "timestamp","t_idx"]),
    "MSRTQps":     (rtq,  TRAFFIC_COLS + ["timestamp","t_idx"]),
    "MSCallGraph": (cg,  ['traceid','timestamp','rpcid','UM','rpctype','DM','interface','rt','t_idx']),
}


### Check Missing Values

In [3]:



for name, (df, num_cols) in tables.items():
    n_total = df.count()
    print(f"\n  {name}: {n_total:,} rows")
 
    # Null counts per column
    null_exprs = [
        F.count(F.when(F.col(c).isNull(), 1)).alias(c)
        for c in df.columns
    ]
    null_pd = df.select(null_exprs).toPandas().T
    null_pd.columns = ["null_count"]
    null_pd["null_pct"] = null_pd["null_count"] / n_total * 100
    null_pd = null_pd[null_pd["null_count"] > 0]
 
    if null_pd.empty:
        print(f"No nulls found")
    else:
        print(null_pd.round(2).to_string())
        


  MSResource: 138,619,760 rows


                    null_count  null_pct
cpu_utilization          16002      0.01
memory_utilization       13033      0.01

  MSRTQps: 936,532 rows
                 null_count  null_pct
HTTP_MCR             570267     60.89
HTTP_RT              570267     60.89
consumerMQ_MCR       316904     33.84
consumerMQ_RT        316904     33.84
consumerRPC_MCR       47735      5.10
consumerRPC_RT        47735      5.10
providerRPC_MCR       63792      6.81
providerRPC_RT        63792      6.81

  MSCallGraph: 506,254,818 rows


[Stage 13:=====================================================>(158 + 2) / 160]

           null_count  null_pct
timestamp           1      0.00
rpcid               2      0.00
UM            6112600      1.21
rpctype             8      0.00
DM                 10      0.00
interface   266008258     52.54
rt                 22      0.00
t_idx               1      0.00


### Handling Missing values

In [6]:
### MSresources table
from pyspark.ml.feature import Imputer

#  Output to temporary columns
imputer = Imputer(
    inputCols=["cpu_utilization", "memory_utilization"],
    outputCols=["cpu_utilization_imputed", "memory_utilization_imputed"],
    strategy="median"
)

res = imputer.fit(res).transform(res)

# Drop originals and rename imputed columns
res = (res
    .drop("cpu_utilization", "memory_utilization")
    .withColumnRenamed("cpu_utilization_imputed", "cpu_utilization")
    .withColumnRenamed("memory_utilization_imputed", "memory_utilization")
)


### MSRTQps
rtq = rtq.fillna(
    0,
    subset=[
        "providerRPC_MCR", "providerRPC_RT",
        "consumerRPC_MCR", "consumerRPC_RT",
        "HTTP_MCR", "HTTP_RT",
        "consumerMQ_MCR", "consumerMQ_RT"
    ]
)

### CallGraph

cg = cg.fillna({
    "UM": "EXTERNAL",
    "interface": "NO_INTERFACE"
})
# a request first enters the system, 
# there may be no upstream microservice because the caller is 
# outside  microservice architecture.

cg = cg.dropna(subset=[

    "timestamp",
    "rpcid",
    "rpctype",
    "DM",
    "rt",
    "t_idx"
])






In [7]:
tables = {
    "MSResource":  (res,  ["cpu_utilization","memory_utilization",
                            "timestamp","t_idx"]),
    "MSRTQps":     (rtq,  TRAFFIC_COLS + ["timestamp","t_idx"]),
    "MSCallGraph": (cg,  ['traceid','timestamp','rpcid','UM','rpctype','DM','interface','rt','t_idx']),
}

for name, (df, num_cols) in tables.items():
    n_total = df.count()
    print(f"\n  {name}: {n_total:,} rows")
 
    # Null counts per column
    null_exprs = [
        F.count(F.when(F.col(c).isNull(), 1)).alias(c)
        for c in df.columns
    ]
    null_pd = df.select(null_exprs).toPandas().T
    null_pd.columns = ["null_count"]
    null_pd["null_pct"] = null_pd["null_count"] / n_total * 100
    null_pd = null_pd[null_pd["null_count"] > 0]
 
    if null_pd.empty:
        print(f"No nulls found")
    else:
        print(null_pd.round(2).to_string())


  MSResource: 138,619,760 rows


No nulls found

  MSRTQps: 936,532 rows
No nulls found



  MSCallGraph: 506,254,796 rows


[Stage 47:=====================================================>(159 + 1) / 160]

No nulls found


In [5]:
# !uv pip install torch==2.4.1 --index-url https://download.pytorch.org/whl/cu124

### Negative Values audit

In [8]:
for name, (df, num_cols) in tables.items():
    for col in num_cols:
        if col not in df.columns: continue
        n_neg = df.filter(F.col(col) < 0).count()
        if n_neg > 0:
            print(f"    {name}.{col}: {n_neg:,} negative values")

    MSResource.memory_utilization: 425 negative values


    MSCallGraph.rt: 112,612,978 negative values


In [10]:
# RT is recorded as positive in UM and negative in DM
# Absolute response time for modeling/metrics
cg = cg.withColumn("rt_abs", F.abs(F.col("rt")))

# Explicit direction flag (in case you need it as a feature)
cg = cg.withColumn(
    "rt_direction",
    F.when(F.col("rt") > 0, "UM")   # upstream caller
     .when(F.col("rt") < 0, "DM")   # downstream callee
     .otherwise("unknown")
)

res = res.withColumn(
    "memory_utilization",
    F.when(F.col("memory_utilization") < 0, 0.0)
     .otherwise(F.col("memory_utilization"))
)

#Keep sign colmn , add rt_abs columns rt_direction

In [12]:
tables = {
    "MSResource":  (res,  ["cpu_utilization","memory_utilization",
                            "timestamp","t_idx"]),
    "MSRTQps":     (rtq,  TRAFFIC_COLS + ["timestamp","t_idx"]),
    "MSCallGraph": (cg,  ['traceid','timestamp','rpcid','UM','rpctype','DM','interface','rt','t_idx']),
}

### Zero inflatioon audit

In [7]:
# for name, (df, num_cols) in tables.items():
#     n_total = df.count()
#     for col in num_cols:
#         if col not in df.columns: continue
#         n_zero = df.filter(F.col(col) == 0).count()
#         pct    = n_zero / n_total * 100
#         if pct > 30:
#             print(f" {name}.{col}: {n_out:,} zeros")

### Timestamp validity audit

In [13]:


for name, (df, _) in tables.items():
    if "timestamp" not in df.columns: continue
    n_neg_ts = df.filter(F.col("timestamp") < 0).count()
    n_null_ts = df.filter(F.col("timestamp").isNull()).count()
    max_ts    = df.agg(F.max("timestamp")).collect()[0][0]
    min_ts    = df.agg(F.min("timestamp")).collect()[0][0]
    print(f"  {name}: ts range [{min_ts}, {max_ts}]  "
          f"neg={n_neg_ts}  null={n_null_ts}")
    if n_neg_ts > 0:
        print(f" {name}.{col}: {n_neg_tst:,} droprows")

  MSResource: ts range [0, 43170000]  neg=0  null=0
  MSRTQps: ts range [0, 43200000]  neg=0  null=0


[Stage 117:====================================================>(159 + 1) / 160]

  MSCallGraph: ts range [2, 43200000]  neg=0  null=0


###  Check empty strings

In [16]:
for name, (df, _) in tables.items():
    str_cols = [f.name for f in df.schema.fields
                if isinstance(f.dataType, T.StringType)]
    for col in str_cols:
        n_empty  = df.filter(
            F.col(col).isNull() | (F.trim(F.col(col)) == "")
        ).count()
        if n_empty > 0:
            print(f" {name}.{col}: {n_empty:,} empty_strings")
        if n_empty==0:
            print(f"no empty strings values {col}")

no empty strings values msname


no empty strings values msinstanceid


no empty strings values nodeid
no empty strings values msname


no empty strings values traceid


no empty strings values rpcid


...

no empty strings values rpctype


no empty strings values DM


no empty strings values interface


[Stage 183:===================================================> (156 + 4) / 160]

no empty strings values rt_direction


### Service name consistency audit

In [20]:
res_svcs  = set(res.select("msname").distinct()
                   .toPandas()["msname"].tolist())
rtq_svcs  = set(rtq.select("msname").distinct()
                   .toPandas()["msname"].tolist())
cg_dm     = set(cg.select("DM").distinct()
                  .toPandas()["DM"].tolist())
cg_um     = set(cg.select("UM").distinct()
                  .toPandas()["UM"].tolist())
cg_svcs   = cg_dm | cg_um
 
only_res  = res_svcs - rtq_svcs - cg_svcs
only_rtq  = rtq_svcs - res_svcs
only_cg   = cg_svcs  - res_svcs
 
print(f"  Services in MSResource only        : {len(only_res):,}")
print(f"  Services in MSRTQps only           : {len(only_rtq):,}")
print(f"  Services in callgraph only         : {len(only_cg):,}")
print(f"  Services in ALL three tables       : "
      f"{len(res_svcs & rtq_svcs & cg_svcs):,}")
 

[Stage 267:===================================================> (156 + 4) / 160]

  Services in MSResource only        : 0
  Services in MSRTQps only           : 0
  Services in callgraph only         : 15,187
  Services in ALL three tables       : 1,266


###  Exreme value audit

In [17]:
for name, (df, num_cols) in tables.items():
    valid_cols = [c for c in num_cols if c in df.columns]
    if not valid_cols:
        continue

    print(f"\n  {name}")

    # ── Step 1: Compute ALL column stats in ONE job ────────────────
    stat_exprs = []
    for col in valid_cols:
        stat_exprs += [
            F.percentile_approx(col, 0.25).alias(f"{col}__q1"),
            F.percentile_approx(col, 0.75).alias(f"{col}__q3"),
            F.percentile_approx(col, 0.99).alias(f"{col}__p99"),
        ]

    stats_row = df.agg(*stat_exprs).collect()[0]  # single job

    # ── Step 2: Compute ALL outlier counts in ONE job ──────────────
    outlier_exprs = {}
    for col in valid_cols:
        q1 = stats_row[f"{col}__q1"]
        q3 = stats_row[f"{col}__q3"]

        if q1 is None or q3 is None:
            print(f"{col}: skipped — all nulls")
            continue

        iqr = q3 - q1
        if iqr == 0:
            print(f"{col}: skipped — IQR=0")
            continue

        upper = q3 + 3 * iqr
        outlier_exprs[col] = (
            upper,
            stats_row[f"{col}__p99"],
            F.count(F.when(F.col(col) > upper, 1)).alias(f"{col}__n_out")
        )

    if not outlier_exprs:
        continue

    # Single job for all outlier counts
    count_exprs = [v[2] for v in outlier_exprs.values()]
    counts_row  = df.agg(*count_exprs).collect()[0]  # single job

    # ── Step 3: Report ─────────────────────────────────────────────
    for col, (upper, p99, _) in outlier_exprs.items():
        n_out = counts_row[f"{col}__n_out"]
        if n_out > 0:
            print(f" {col}: {n_out:,} outliers | upper={upper:.2f} p99={p99:.2f}")
        else:
            print(f" {col}: no outliers")


  MSResource


    cpu_utilization: 76,353 outliers | upper=0.73 p99=0.61
    ✓ memory_utilization: no outliers
    ✓ timestamp: no outliers
    ✓ t_idx: no outliers

  MSRTQps


26/05/05 10:45:30 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

    providerRPC_MCR: 62,644 outliers | upper=234.06 p99=680.10
    providerRPC_RT: 69,115 outliers | upper=118.82 p99=344.27
    consumerRPC_MCR: 49,580 outliers | upper=232.50 p99=490.26
    consumerRPC_RT: 53,566 outliers | upper=75.20 p99=255.49
    HTTP_MCR: 114,475 outliers | upper=2.94 p99=160.84
    HTTP_RT: 114,475 outliers | upper=2.94 p99=160.84
    consumerMQ_MCR: 86,175 outliers | upper=53.29 p99=497.36
    consumerMQ_RT: 86,175 outliers | upper=53.29 p99=497.36
    ✓ timestamp: no outliers
    ✓ t_idx: no outliers

  MSCallGraph


    ⚠️  traceid: skipped — all nulls
    ⚠️  UM: skipped — all nulls
    ⚠️  rpctype: skipped — all nulls
    ⚠️  DM: skipped — all nulls
    ⚠️  interface: skipped — all nulls


[Stage 195:====================================================>(159 + 1) / 160]

    ✓ timestamp: no outliers
    rpcid: 670,917 outliers | upper=0.40 p99=9.15
    rt: 69,415,732 outliers | upper=4.00 p99=262.00
    ✓ t_idx: no outliers


In [19]:
output_path = "data_clean"   


res.write \
   .mode("overwrite") \
   .parquet(f"{output_path}/MSResource_clean")

rtq.write \
   .mode("overwrite") \
   .parquet(f"{output_path}/MSRTQps_clean")

cg.write \
   .mode("overwrite") \
   .parquet(f"{output_path}/MSCallGraph_clean")

print(" All datasets saved.")

[Stage 202:====================================================>(159 + 1) / 160]

 All datasets saved.


In [20]:
res.columns

['msname',
 'msinstanceid',
 'nodeid',
 'timestamp',
 't_idx',
 'cpu_utilization',
 'memory_utilization']

In [21]:
rtq.columns

['timestamp',
 'msname',
 'HTTP_MCR',
 'HTTP_RT',
 'consumerMQ_MCR',
 'consumerMQ_RT',
 'consumerRPC_MCR',
 'consumerRPC_RT',
 'providerRPC_MCR',
 'providerRPC_RT',
 't_idx']

In [22]:
cg.columns

['traceid',
 'timestamp',
 'rpcid',
 'UM',
 'rpctype',
 'DM',
 'interface',
 'rt',
 't_idx',
 'rt_abs',
 'rt_direction']

In [24]:
cg.head(10)

[Row(traceid='0b52059815919297894615000e4a94', timestamp=6189524, rpcid='0.1.1.2.1.40', UM='35114acfb54c54fb9618f23cd28bbc57c765f597df140977d7030dcc52775ed4', rpctype='mc', DM='431a07b20e43caa33f909b6d5d82c2c9db736acb47307de11a51324d029c9079', interface='NO_INTERFACE', rt=0.0, t_idx=206, rt_abs=0.0, rt_direction='unknown'),
 Row(traceid='0b144db915919351673291000e9b73', timestamp=11567332, rpcid='0.1.1', UM='(?)', rpctype='http', DM='95a6f7f8345e2eca31ee74ddc19d547e7fc0f5c8e65772d7b08a68eb5214dc44', interface='b5d09f361bd1ea282705b61915e2965e10cbd7fbe367b4b830f64136141c8622', rt=-359.0, t_idx=385, rt_abs=359.0, rt_direction='DM'),
 Row(traceid='0b13393a15919279916136000e84fc', timestamp=4391689, rpcid='0.1.1.2.14.1', UM='3cab0a98767379fbcd059750c07ec42dbd0374cb114e0f8dc3a75bbaf4eed7d3', rpctype='rpc', DM='75e56c8fbb9336eb4dd40f5f609d5344203d374d73fd0b8d2bce19f754070f6a', interface='dd1969d2e5b020dfdb88c223417a901d9727663879975982b23de4f5b7746990', rt=199.0, t_idx=146, rt_abs=199.0, rt_